# Run the full ETL pipeline (Option 1)
`public` (source) -> `staging` -> `transformed` -> `clean`

**Before you start:** put `rules.sql`, `extract.sql`, `transform.sql`, `load.sql` and `schema.sql` in `SQL_DIR`, and the five CSV files in `DATA_DIR`.
Run the cells top to bottom. Step 2 is destructive for `public` (it reloads it from the CSVs), so it is guarded by `CONFIRM_RESET`.

In [7]:
from pathlib import Path
from io import StringIO
import pandas as pd
from sqlalchemy import text
from connectdb import get_engine

engine = get_engine()

SQL_DIR  = Path("sql")    # folder with rules.sql, extract.sql, transform.sql, load.sql, schema.sql
DATA_DIR = Path("data")   # folder with fact_sales.csv, dim_*.csv

CONFIRM_RESET   = True   # set True to reload public from the CSVs (step 2)
LOAD_RAW_SALES  = True    # optional: load original sale categories, only used for the audit table

def run_file(path):
    raw = engine.raw_connection()
    try:
        raw.cursor().execute(Path(path).read_text())
        raw.commit()
    finally:
        raw.close()

def query(sql):
    return pd.read_sql(text(sql), engine)
print("connected:", query("select current_database() as db").iloc[0, 0])

connected: postgres


## Step 2: reset `public` from the CSVs
This removes the earlier notebook edits (category update, customer deletes, `customer_dedup_map`, `fact_sales_clean`) and the old pipeline schemas.

In [ ]:
if not CONFIRM_RESET:
    raise SystemExit("Set CONFIRM_RESET = True to reload public from the CSVs.")

raw = engine.raw_connection()
cur = raw.cursor()
cur.execute("""
    DROP SCHEMA IF EXISTS clean, transformed, staging, etl CASCADE;
    DROP VIEW  IF EXISTS public.fact_sales_clean;
    DROP TABLE IF EXISTS public.customer_dedup_map, public.fact_sales, public.dim_store,
                         public.dim_customer, public.dim_product, public.dim_date CASCADE;
""")
raw.commit()
cur.execute((DATA_DIR / "schema.sql").read_text())
for table in ["dim_date", "dim_product", "dim_customer", "dim_store", "fact_sales"]:
    with open(DATA_DIR / f"{table}.csv") as fh:
        cur.copy_expert(f"COPY public.{table} FROM STDIN WITH CSV HEADER", fh)
raw.commit()
raw.close()
query("""select 'fact_sales' t, count(*) n from public.fact_sales union all
         select 'dim_product', count(*) from public.dim_product union all
         select 'dim_customer', count(*) from public.dim_customer""")

InvalidSchemaName: no schema has been selected to create in
LINE 3: CREATE TABLE IF NOT EXISTS dim_date (
                                   ^


## Step 3: install the rules and the three stage functions

In [ ]:
for name in ["rules", "extract", "transform", "load"]:
    run_file(SQL_DIR / f"{name}.sql")
    print("installed", name)
query("select count(*) as product_names_mapped from etl.product_category_map")

## Step 4: run extract -> transform -> load

In [ ]:
raw = engine.raw_connection()
conn = getattr(raw, "driver_connection", None) or raw.connection
cur = raw.cursor()

cur.execute("SELECT etl.extract()"); raw.commit()

if LOAD_RAW_SALES:
    df = pd.read_csv(DATA_DIR / "fact_sales.csv", usecols=["sale_id", "category"])
    buf = StringIO(); df.to_csv(buf, index=False); buf.seek(0)
    cur.copy_expert("COPY staging.raw_sales (sale_id, category) FROM STDIN WITH CSV HEADER", buf)
    raw.commit()

del conn.notices[:]
cur.execute("SELECT etl.transform()"); raw.commit()
cur.execute("SELECT etl.load()");      raw.commit()

for n in conn.notices:
    if "Issue" in n:
        print(n.strip())
raw.close()
print("pipeline finished")

## Step 5: validate

In [ ]:
query("""
select 'dim_customer rows (expect 1995)'          as check_name, count(*)::text as value from clean.dim_customer
union all select 'fact_sales rows (expect 60000)', count(*)::text from clean.fact_sales
union all select 'fact_sales has own category column (expect false)',
       (count(*) > 0)::text from information_schema.columns
       where table_schema = 'clean' and table_name = 'fact_sales' and column_name = 'category'
union all select 'view category differs from dim_product (expect 0)', count(*)::text
       from clean.v_fact_sales v join clean.dim_product p using (product_key)
       where v.category is distinct from p.category
union all select 'products with wrong category vs name map (expect 0)', count(*)::text
       from clean.dim_product p join etl.product_category_map m on m.product_name = p.product_name
       where p.category <> m.category
union all select 'negative-margin products (expect 0)', count(*)::text from clean.dim_product where unit_cost > list_price
union all select 'month_name mismatches (expect 0)', count(*)::text
       from clean.dim_date where month_name <> to_char(full_date, 'FMMonth')
union all select 'revenue mismatch percent (documented)',
       round(100.0 * count(*) filter (where revenue_mismatch) / count(*), 2)::text from clean.fact_sales
""")

In [ ]:
query("""
select category, count(*) as sales, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
from clean.v_fact_sales group by category order by sales desc
""")

In [ ]:
query("select category, count(*) as products from clean.dim_product group by category order by products desc")